# Task 1: repository and Kaggle setup

Run on Kaggle with **Accelerator: GPU T4 x2** and **Internet: on**. The notebook clones the project,
installs dependencies without touching Kaggle's preinstalled CUDA PyTorch, checks that two T4 GPUs are
visible, prints package versions and runs the public adapter tests. Everything is written under
`/kaggle/working` so a saved notebook version keeps it.

In [1]:
REPO_URL = "https://github.com/AyanAnees326/small-model-alignment.git"
BRANCH = "main"
WORKDIR = "/kaggle/working/small-model-alignment"

!git clone --depth 1 --branch {BRANCH} {REPO_URL} {WORKDIR}
%cd {WORKDIR}
!git log -1 --oneline

Cloning into '/kaggle/working/small-model-alignment'...
remote: Enumerating objects: 124, done.
remote: Counting objects: 100% (124/124), done.
remote: Compressing objects: 100% (99/99), done.
remote: Total 124 (delta 33), reused 83 (delta 21), pack-reused 0 (from 0)
Receiving objects: 100% (124/124), 3.20 MiB | 4.64 MiB/s, done.
Resolving deltas: 100% (33/33), done.
/kaggle/working/small-model-alignment
343b0c3 (grafted, HEAD -> main, origin/main, origin/HEAD) Add A/B-swapped pair files for a second, order-debiasing judge pass; register LR ablation comparison


Install exactly as the manual recommends (§5.2): no PyTorch reinstall, drop the stale `torchao`, install the repo with `--no-deps`. `trl`/`peft` are only needed for Task 5.

In [2]:
!python -m pip uninstall -y -q torchao || true
!python -m pip install -q "transformers>=4.43.0,<5" "datasets>=2.18,<5" huggingface_hub accelerate tqdm pyyaml sentencepiece safetensors pytest
!python -m pip install -q -e . --no-deps
!python -m pip install -q "trl>=0.19" peft --no-deps

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 98.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 26.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 89.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for small-model-alignment-starter (pyproject.toml) ..

In [3]:
!nvidia-smi

Sat Oct 17 10:24:39 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   51C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [4]:
import torch, transformers, datasets

print("torch       ", torch.__version__, "| CUDA", torch.version.cuda)
print("transformers", transformers.__version__)
print("datasets    ", datasets.__version__)
print("GPUs        ", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
assert torch.cuda.device_count() == 2, "expected the GPU T4 x2 accelerator"

torch        2.11.0+cu128 | CUDA 12.8
transformers 4.57.6
datasets     4.8.5
GPUs         ['Tesla T4', 'Tesla T4']


In [ ]:
!PYTHONPATH=. python -m pytest -q tests/test_public_adapter_smoke.py

## Optional: full pipeline on Kaggle

The submitted runs were trained locally (see the report, §14.3). To reproduce them on T4 x2 instead, run
the stages below; `alignment.train` puts the policy on `cuda:0` and the frozen reference on `cuda:1`
automatically, and `--resume` continues from `<out>/last/state.pt` after a disconnect.

```bash
python -m alignment.data                                   # data/processed/*.jsonl + outputs/data_manifest.json
bash scripts/run_local_pipeline.sh                         # SFT -> generations -> 2x DPO -> LoRA-DPO -> pairs -> plots
```